# Track B — Fase 3b: Retrain 2-backbone (so400m + DINOv3) di folds_v2

**BDC Satria Data 2026** | Uji ulang fusi so400m + DINOv3 di data bersih — dengan perbaikan metodologi yang bikin v1 nolak concat.

| | |
|---|---|
| Backbone | `siglip2so400m` (dim 1152) + `dinov3vitl` (dim ~1024), keduanya frozen, non-TTA |
| Head | kNN (k=15, cosine, weights=distance) |
| Split | kolom `fold` dari **folds_v2** (apa adanya, tak di-stratify ulang) |
| Acuan | so400m only CV v1 = mean 0.9901 / min 0.9896 / std 0.0005 · **test asli 98.5** |

> **Kenapa dicoba lagi padahal v1 nolak concat:** v1 nolak karena gain +0.0006 (noise) & min/std memburuk. Tapi concat v1 kemungkinan **naif** — embedding di-concat mentah. so400m (norm lebih besar) mendominasi jarak cosine, DINOv3 nyaris tak menyumbang. Itu bug metodologi, bukan bukti DINOv3 tak berguna.

> **Motivasi nyata:** CV 0.9901 tapi test 0.985 → gap ~0.5pt. DINOv3 (self-supervised) error-nya orthogonal ke SigLIP (image-text). Menambah backbone orthogonal itu obat klasik untuk gap CV↔test — **asal fusinya benar**.

> **⚠️ Aturan keras:** pemenang dipilih dari **OOF**, BUKAN dari skor test 0.985. Memilih komposisi berdasarkan test = overfit leaderboard + langgar aturan panitia (test hanya untuk prediksi akhir). Skor test cuma konteks, bukan kriteria seleksi.

**4 kandidat dibandingkan:** (1) so400m only · (2) concat mentah ≈ cara v1 · (3) concat + L2-norm per-backbone (fix) · (4) ensemble prob-average.

---
## 🔧 SETUP

In [ ]:
# Cell 1 — Repo + Drive + dependensi (CPU, tanpa GPU)
import os
if not os.path.exists('/content/satria-data-bdcugm02'):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
else:
    !git -C /content/satria-data-bdcugm02 pull

!pip install -q scikit-learn

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ setup siap')

In [ ]:
# Cell 2 — Path + pilih varian DINOv3. Verifikasi file WAJIB ada.
import os, sys
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
from config import CFG

SIGLIP = 'siglip2so400m'
DINO   = 'dinov3vitl'          # ganti ke 'dinov3cnxb' kalau mau uji varian ConvNeXt

EMB      = CFG.embeddings_dir
FOLDS_V1 = CFG.folds_csv
FOLDS_V2 = os.path.join(os.path.dirname(CFG.folds_csv), 'folds_v2.csv')
SIG_TRAIN  = os.path.join(EMB, f'{SIGLIP}_train.npy')
DINO_TRAIN = os.path.join(EMB, f'{DINO}_train.npy')

checks = [(FOLDS_V1,'folds v1'), (FOLDS_V2,'folds v2'),
          (SIG_TRAIN,'emb siglip'), (DINO_TRAIN,'emb dino')]
ok = True
for p, tag in checks:
    exists = os.path.exists(p); ok = ok and exists
    print(('✅' if exists else '❌ HILANG'), f'{tag:12s}', p)
assert ok, 'Ada file wajib hilang — selesaikan dulu'
print(f'\nSIGLIP={SIGLIP}  DINO={DINO}  seed={CFG.seed}')

---
## 🔗 ALIGN — reindex KEDUA embedding v1 → urutan folds_v2 (join lewat filepath)

In [ ]:
# Cell 3 — Pemetaan filepath → posisi v1 + diagnostik drop/relabel (sama seperti retrain 1-backbone)
import pandas as pd, numpy as np

v1 = pd.read_csv(FOLDS_V1)
v2 = pd.read_csv(FOLDS_V2)
for col in ['filepath', 'label', 'fold']:
    assert col in v1.columns and col in v2.columns, f'kolom {col!r} hilang'
assert v1['filepath'].is_unique and v2['filepath'].is_unique, 'filepath tidak unik'

pos_of = {fp: i for i, fp in enumerate(v1['filepath'])}
missing = [fp for fp in v2['filepath'] if fp not in pos_of]
assert not missing, f'{len(missing)} filepath v2 tak ada di v1 (contoh {missing[:3]}) — cek format path Track A'
pos = v2['filepath'].map(pos_of).to_numpy()

v1_label = dict(zip(v1['filepath'], v1['label']))
v1_fold  = dict(zip(v1['filepath'], v1['fold']))
n_drop     = len(v1) - len(v2)
relabeled  = int(sum(v1_label[fp] != lb for fp, lb in zip(v2['filepath'], v2['label'])))
fold_moved = int(sum(v1_fold[fp]  != fd for fp, fd in zip(v2['filepath'], v2['fold'])))

print(f'v1: {len(v1)}  v2: {len(v2)}  | DROP {n_drop} ({n_drop/len(v1)*100:.2f}%)  RELABEL {relabeled}')
assert fold_moved == 0, f'{fold_moved} survivor pindah fold — langgar aturan Track A, v2 tak sebanding v1'
print('✅ fold survivor tak berubah')
if n_drop / len(v1) > 0.03:
    print(f'⚠️  DROP > cap 3% — konfirmasi ke Track A')

In [ ]:
# Cell 4 — Load kedua embedding, reindex ke urutan v2, siapkan y & fold
def load_reindex(path, name):
    A = np.load(path)
    assert len(A) == len(v1), f'{name}: emb {len(A)} != folds_v1 {len(v1)} — kontrak alignment putus'
    assert np.isfinite(A).all(), f'{name}: ada NaN/Inf (DINOv3 harus fp32!)'
    return A[pos]

Xs = load_reindex(SIG_TRAIN,  'siglip')
Xd = load_reindex(DINO_TRAIN, 'dino')
y    = v2['label'].to_numpy()
fold = v2['fold'].to_numpy()
assert len(Xs) == len(Xd) == len(y) == len(fold) == len(v2)

# Sanity align lewat filepath (kedua backbone dipetakan pos yang sama)
rng = np.random.default_rng(CFG.seed)
for i in rng.choice(len(v2), size=min(5, len(v2)), replace=False):
    fp = v2['filepath'].iloc[i]
    assert np.array_equal(Xs[i], np.load(SIG_TRAIN)[pos_of[fp]]), f'align siglip gagal @ {i}'
print('✅ align terverifikasi')
print(f'Xs {Xs.shape} (siglip)  |  Xd {Xd.shape} (dino)')
print('dist kelas y:', dict(zip(*[a.tolist() for a in np.unique(y, return_counts=True)])))

---
## 🚀 CV — 4 kandidat fusi

In [ ]:
# Cell 5 — Helper CV. Semua pakai kolom fold folds_v2, kNN terkunci (k=15, cosine, distance).
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

K, LABELS = 15, [0, 1, 2]

def l2norm(A, eps=1e-12):
    # Normalisasi per-baris → tiap backbone bobot setara saat concat (kunci fix concat)
    return A / (np.linalg.norm(A, axis=1, keepdims=True) + eps)

def _fill(oof_slice_idx, oof, clf, proba):
    for j, c in enumerate(clf.classes_):
        oof[oof_slice_idx, int(c)] = proba[:, j]

def cv_knn(Xin, y, fold, k=K):
    # Feature-level: satu kNN di atas Xin (bisa single / concat)
    oof = np.zeros((len(y), 3)); per = []
    for f in sorted(np.unique(fold)):
        tr, va = fold != f, fold == f; idx = np.where(va)[0]
        clf = KNeighborsClassifier(n_neighbors=k, metric='cosine', weights='distance').fit(Xin[tr], y[tr])
        _fill(idx, oof, clf, clf.predict_proba(Xin[va]))
        per.append(f1_score(y[va], oof[va].argmax(1), labels=LABELS, average='macro', zero_division=0.0))
    return oof, np.array(per)

def cv_ensemble(list_X, y, fold, k=K):
    # Decision-level: kNN terpisah per backbone, rata-ratakan probabilitas (bobot setara)
    oof = np.zeros((len(y), 3)); per = []
    for f in sorted(np.unique(fold)):
        tr, va = fold != f, fold == f; idx = np.where(va)[0]
        acc = np.zeros((int(va.sum()), 3))
        for Xin in list_X:
            clf = KNeighborsClassifier(n_neighbors=k, metric='cosine', weights='distance').fit(Xin[tr], y[tr])
            tmp = np.zeros((int(va.sum()), 3))
            for j, c in enumerate(clf.classes_): tmp[:, int(c)] = clf.predict_proba(Xin[va])[:, j]
            acc += tmp
        acc /= len(list_X); oof[idx] = acc
        per.append(f1_score(y[va], acc.argmax(1), labels=LABELS, average='macro', zero_division=0.0))
    return oof, np.array(per)

print('helper siap')

In [ ]:
# Cell 6 — Jalankan 4 kandidat (~beberapa menit di CPU; concat lebih lambat karena dim besar)
candidates = {}
print('running sig_only ...');    candidates['sig_only']   = cv_knn(Xs, y, fold)
print('running concat_raw ...');  candidates['concat_raw'] = cv_knn(np.hstack([Xs, Xd]), y, fold)
print('running concat_l2 ...');   candidates['concat_l2']  = cv_knn(np.hstack([l2norm(Xs), l2norm(Xd)]), y, fold)
print('running ens_avg ...');     candidates['ens_avg']    = cv_ensemble([Xs, Xd], y, fold)

rows = []
for name, (oof, per) in candidates.items():
    rows.append(dict(kandidat=name, mean=per.mean(), min=per.min(), std=per.std()))
tab = pd.DataFrame(rows).sort_values('mean', ascending=False).reset_index(drop=True)
print()
print(tab.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

---
## 📊 VERDICT — pilih dari OOF, bukan test

In [ ]:
# Cell 7 — Bandingkan tiap kandidat vs so400m-only DAN vs v1 (0.9901). Aturan anti-overfit sama.
base = dict(zip(['mean','min','std'], [candidates['sig_only'][1].mean(),
                                       candidates['sig_only'][1].min(),
                                       candidates['sig_only'][1].std()]))
V1_REF = 0.9901
print(f"acuan sig_only (folds_v2): mean={base['mean']:.4f} min={base['min']:.4f} std={base['std']:.4f}")
print(f'acuan so400m-only CV v1  : mean={V1_REF:.4f}   |   test asli = 0.985\n')

def verdict(name):
    _, per = candidates[name]
    m, mn, sd = per.mean(), per.min(), per.std()
    dme, dmi = m - base['mean'], mn - base['min']
    if dme > 0.002 and dmi >= 0:   tag = '✅ NAIK nyata (mean↑, min tak turun) — kandidat kuat'
    elif abs(dme) < 0.002:         tag = '➖ dalam noise vs sig_only — pertimbangkan diversity, TAPI jangan pilih dari test'
    elif dme > 0 and dmi < 0:      tag = '⚠️ mean↑ tapi min↓ — TOLAK (satu fold beruntung)'
    else:                          tag = '❌ turun vs sig_only'
    print(f'{name:12s} mean={m:.4f} min={mn:.4f} std={sd:.4f} | Δmean={dme:+.4f} Δmin={dmi:+.4f}  {tag}')

for name in ['concat_raw', 'concat_l2', 'ens_avg']:
    verdict(name)

print('\nCatatan baca:')
print('- concat_raw ≈ sig_only  → konfirmasi v1: concat mentah memang DINOv3 ketimbun.')
print('- concat_l2 / ens_avg naik → fusi yang BENAR menolong; diversity nyata.')
print('- semua flat → DINOv3 memang tak menambah untuk task ini, sig_only tetap pilihan.')
print('- Keputusan final tetap lewat Track C (weight search di OOF), bukan skor test.')

In [ ]:
# Cell 8 — Per-kelas F1 kandidat terbaik-OOF (cek klaim reviewer Organic↔Recyclable)
from sklearn.metrics import confusion_matrix
best_name = max(candidates, key=lambda n: candidates[n][1].mean())
best_oof  = candidates[best_name][0]
pred = best_oof.argmax(1)
print('kandidat mean-tertinggi (OOF):', best_name)
print('per-kelas F1 (0=Recyclable,1=Electronic,2=Organic):',
      np.round(f1_score(y, pred, labels=LABELS, average=None, zero_division=0.0), 4))
print('confusion matrix:')
print(confusion_matrix(y, pred, labels=LABELS))

---
## 💾 HANDOFF — OOF per-backbone + fusi ke Track C (guard anti-overwrite)

In [ ]:
# Cell 9 — Simpan OOF supaya Track C bisa weight-search sendiri. File existing = FileExistsError.
import json
OUT = CFG.save_dir

to_save = {
    f'oof_{DINO}_knn_v2.npy'                     : cv_knn(Xd, y, fold)[0],   # DINOv3 tunggal (utk weight search)
    f'oof_concat_{SIGLIP}_{DINO}_l2_knn_v2.npy'  : candidates['concat_l2'][0],
    f'oof_ensavg_{SIGLIP}_{DINO}_knn_v2.npy'     : candidates['ens_avg'][0],
}
# Catatan: OOF so400m tunggal sudah disimpan notebook retrain 1-backbone (jangan tulis ulang).

for fname in to_save:
    p = os.path.join(OUT, fname)
    if os.path.exists(p):
        raise FileExistsError(f'{p} sudah ada — jangan overwrite. Hapus manual kalau memang mau regenerate.')

for fname, arr in to_save.items():
    np.save(os.path.join(OUT, fname), arr)
    print('tersimpan:', fname, arr.shape)

meta = dict(
    siglip=SIGLIP, dino=DINO, head='knn', k=15, metric='cosine', weights='distance', tta=False,
    folds='folds_v2.csv', seed=int(CFG.seed), n=int(len(y)),
    dropped_from_v1=int(n_drop), relabeled=int(relabeled),
    cv={name: dict(mean=float(per.mean()), min=float(per.min()), std=float(per.std()))
        for name, (_, per) in candidates.items()},
    label_order=[0, 1, 2],
    align_note='baris ke-i tiap oof = baris ke-i folds_v2.csv (urut sama).',
)
mp = os.path.join(OUT, f'oof_fusi_{SIGLIP}_{DINO}_meta.json')
if os.path.exists(mp): raise FileExistsError(f'{mp} sudah ada')
json.dump(meta, open(mp, 'w'), indent=2)
print('meta:', mp)
print('\nSIAP KE TRACK C — kirim OOF per-backbone + fusi; biar weight search yang putuskan bobot final.')

---
## Langkah berikutnya

1. **Baca Cell 6–7.** Kalau `concat_l2` atau `ens_avg` naik nyata vs `sig_only` (mean↑, min tak turun) → fusi layak masuk kandidat panen. Kalau semua flat → DINOv3 tak menambah, `sig_only` tetap.
2. **Serahkan OOF per-backbone ke Track C**, bukan cuma fusi. Track C punya modul weight search (nested validation) — biarkan dia cari bobot optimal so400m:dino, lebih baik daripada mengunci 50:50 di sini.
3. **Gap CV↔test (0.9901 vs 0.985):** kalau fusi menaikkan OOF sedikit tapi konsisten antar-fold, itu justru sinyal bagus untuk gap — diversity mengurangi overfit ke distribusi train. Tapi buktinya tetap dari submission panen nanti, bukan dari nebak.
4. **JANGAN** ganti komposisi hanya karena test 0.985. Itu 1 titik data dari 1 submission; menjadikannya kriteria seleksi = overfit leaderboard. OOF 5-fold lebih dapat dipercaya untuk memilih.

> Ganti `DINO='dinov3cnxb'` di Cell 2 lalu jalankan ulang kalau mau bandingkan varian ConvNeXt-B. Nama file output sudah otomatis ikut varian, jadi tak bentrok.